<!-- SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
SPDX-License-Identifier: OpenMDW-1.1 -->

# Cosmos3.5 Action: Inverse dynamics with vLLM-Omni

Tests recovery from a video, history exclusion, measured/predicted initial-state rows, checkpoint-derived and overridden schedules, sparse channels, prompt rendering, and invalid requests. The default uses a camera-only 59D mask. The old direct 9D/framewise plot is replaced with masked, denormalized anchored poses.

An already deployed, compatible Cosmos3.5 action model is required. There are no deployment or weight-download cells. Use the endpoint's existing guardrail configuration. These notebooks run all cases, including expected failures, and preserve each run in a fresh output directory.

Default inputs are **synthetic protocol probes**, normalized with the framework's canonical global-asinh metric-gripper profile and paired with checked-in observation media. They test structural invariants, not task success or trajectory accuracy. For recorded examples set the fixture variables described in [the regression guide](vllm_omni_regression.md).

## Client dependencies

The framework checkout must be importable. Torch is used only for CPU action normalization/decoding; no model is loaded in the notebook. Install these dependencies in the notebook kernel if needed.

In [ ]:
%pip install numpy requests pillow "av>=18,<19" matplotlib torch loguru rich

## Endpoint, fixtures and output directory

Set `COSMOS3_VLLM_BASE_URL` to the existing server. Optional variables: `COSMOS3_VLLM_API_KEY`, `COSMOS3_VLLM_OUTPUT_ROOT`, `COSMOS3_ACTION_TIMEOUT_S`, `COSMOS3_ACTION_STATS`, `COSMOS3_ACTION_FIXTURE`, and `COSMOS3_CAMERA_FIXTURE`.

The default 736x544 fixture expects checkpoint-derived video shift 3.5 and action shift 1. Use fixture metadata matching your checkpoint if its training shifts differ. The primary request omits both shifts; explicit cases verify the expected defaults. All requests keep a fixed seed and 30 denoising steps.

In [ ]:
SUITE = "id"

import os
import sys
import uuid
from datetime import datetime, timezone
from pathlib import Path

# Resolve from either the checkout root or this notebook's directory.
FRAMEWORK_ROOT = next(
    path
    for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (path / "cosmos_framework").is_dir() and (path / "pyproject.toml").is_file()
)
ACTION_ROOT = FRAMEWORK_ROOT / ".agents/skills/cookbooks/cosmos3/generator/action"
sys.path.insert(0, str(FRAMEWORK_ROOT))
sys.path.insert(0, str(ACTION_ROOT))

import numpy as np
from vllm_omni_regression import (
    ActionClient,
    action_normalizer,
    backend_test_command,
    compare_actions,
    compare_reference,
    decode_actions,
    load_fixture,
    prediction_cases,
    protocol_fixtures,
)

stats_path = os.environ.get("COSMOS3_ACTION_STATS")
normalizer = action_normalizer(FRAMEWORK_ROOT, Path(stats_path) if stats_path else None)
robot, camera = protocol_fixtures(ACTION_ROOT, normalizer)
if os.environ.get("COSMOS3_ACTION_FIXTURE"):
    robot = load_fixture(Path(os.environ["COSMOS3_ACTION_FIXTURE"]).resolve())
if os.environ.get("COSMOS3_CAMERA_FIXTURE"):
    camera = load_fixture(Path(os.environ["COSMOS3_CAMERA_FIXTURE"]).resolve())

base_url = os.environ.get("COSMOS3_VLLM_BASE_URL", "http://localhost:8001")
output_base = Path(os.environ.get("COSMOS3_VLLM_OUTPUT_ROOT", FRAMEWORK_ROOT / "outputs/cosmos35_action_vllm"))
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
client = ActionClient(
    base_url, output_base / SUITE / run_id, timeout_s=float(os.environ.get("COSMOS3_ACTION_TIMEOUT_S", "1800"))
)
if os.environ.get("COSMOS3_VLLM_API_KEY"):
    client.session.headers["Authorization"] = "Bearer " + os.environ["COSMOS3_VLLM_API_KEY"]
models = client.models()
client._save(
    client.output_root / "normalization.json",
    {
        "method": "global_asinh",
        "offset": normalizer.offset.tolist(),
        "scale": normalizer.scale.tolist(),
        "unit": normalizer.unit,
        "pose_convention": "backward_anchored",
    },
)
print("Models:", models)
print("Outputs:", client.output_root)
print("Robot fixture:", robot.provenance)
print("Camera fixture:", camera.provenance)

## Run the regression matrix

Each request and terminal response is saved. Successful cases require finite `[K,59]` actions, matching metadata, zero inactive channels, and preservation of supplied FD/state rows. Video cases must decode to exactly 17 frames at the requested FPS and canvas. Action-only cases require empty video content. Rejections must match the intended validation error; arbitrary failures do not count as passing tests.

In [ ]:
cases = prediction_cases(camera, inverse=True)
for case in cases:
    print(case.name, "reject: " + case.expected_error if case.expected_error else "success")
results = client.run_suite(cases)

## Compare equivalent requests

Equivalent aliases, automatic defaults and explicit matching options must reproduce the normalized action predictions within numerical tolerance. Shared-schedule and changed-action-shift requests are ablations: their shapes and values are validated, but a numerical difference alone would not establish correctness.

In [ ]:
compare_actions(client.output_root, "inverse_dynamics_default", "inverse_dynamics_explicit_schedule")
compare_actions(client.output_root, "inverse_dynamics_default", "inverse_dynamics_raw_prompt_reference")
compare_actions(client.output_root, "inverse_dynamics_json_prompt", "inverse_dynamics_json_alias")

## Inspect the denormalized trajectory

Translations below are already relative to the current observation. Do not cumulatively integrate these anchored translations as framewise deltas. Inactive channels are cleared again after inverse normalization.

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Video, display

fixture = camera
normalized = np.load(client.output_root / "inverse_dynamics_default" / "action.npy", allow_pickle=False)
physical = decode_actions(normalized, fixture, normalizer)
np.save(client.output_root / "inverse_dynamics_default" / "action_physical.npy", physical)
positions = physical[:, 0:3]
fig = plt.figure(figsize=(7, 5))
ax = fig.add_subplot(111, projection="3d")
ax.plot(*positions.T, marker="o")
ax.set(xlabel="X (m)", ylabel="Y (m)", zlabel="Z (m)", title="Anchored predicted/conditioned pose")
plt.show()
video_path = client.output_root / "inverse_dynamics_default" / "video.mp4"
if video_path.exists():
    display(Video(str(video_path), embed=True))

## Optional reference parity

Set `COSMOS3_ACTION_REFERENCE_ROOT` to reference artifacts exported with matching checkpoint, preprocessing, sampling and **initial noise**. A matching seed alone is insufficient across backends. Without references this notebook claims protocol coverage only. See the [reference format](vllm_omni_regression.md#reference-artifacts).

In [ ]:
reference_root = os.environ.get("COSMOS3_ACTION_REFERENCE_ROOT")
if reference_root:
    compare_reference(
        client.output_root, Path(reference_root) / SUITE, [r["name"] for r in results if r["status"] == "passed"]
    )
else:
    print("Cross-backend numerical parity not checked: no reference root configured.")

## Internal guarantees: run in the vLLM-Omni environment

HTTP output does not expose the resolved projectors, component timesteps, mRoPE offsets, rendered tokens, or (currently) history/state/per-row-domain metadata. The focused upstream tests complement this notebook. Configure both variables below to execute them using the backend's own Python environment; no vLLM dependency is added to this kernel. If unset, those guarantees remain explicitly unverified.

- `COSMOS3_VLLM_REPO`: local vLLM-Omni checkout containing your changes.
- `COSMOS3_VLLM_PYTHON`: absolute Python executable for that checkout's test environment.

In [ ]:
import subprocess

backend_repo = os.environ.get("COSMOS3_VLLM_REPO")
backend_python = os.environ.get("COSMOS3_VLLM_PYTHON")
if backend_repo and backend_python:
    repo = Path(backend_repo).resolve()
    command = backend_test_command(repo, backend_python)
    client._save(client.output_root / "backend_command.json", command)
    completed = subprocess.run(
        command, cwd=repo, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=900
    )
    (client.output_root / "backend_tests.log").write_text(completed.stdout)
    print(completed.stdout)
    completed.check_returncode()
else:
    print("Internal-only checks not run. Configure COSMOS3_VLLM_REPO and COSMOS3_VLLM_PYTHON ")
    print("to verify config fallback/projectors, component timesteps, mRoPE offsets, ")
    print("per-row routing, clean history, masking at every step, and exact prompt goldens.")